# SentraGrade: extra post-hoc OOD baselines on the leakage-corrected split

Reviewers expect the three scores of the paper (MSP, energy, prototype distance) to be compared with the standard post-hoc OOD detectors. This notebook adds five of them **without retraining anything**: it reuses the ResNet50 checkpoints of `leakage_split_resnet50.ipynb` (which store the logits and 2048-d features of every train/val/test/held-out image plus the final layer) and the frozen ViT-B/16 and DINOv2 linear heads of `vit_dinov2_probe.ipynb`.

| Score | Idea | Uses |
|---|---|---|
| MSP, Energy, Prototype | the paper's three scores (recomputed as a check) | logits / features |
| **MaxLogit** (Hendrycks et al., 2022) | negative largest raw logit | logits |
| **Mahalanobis** (Lee et al., 2018) | distance to the nearest class mean under a shared covariance of training features | features |
| **KNN** (Sun et al., 2022) | negative cosine similarity to the $k$-th nearest training feature, $k=50$ | features |
| **ReAct** (Sun et al., 2021) | energy after clipping features at the 90th percentile of training activations | features + last layer |
| **ViM** (Wang et al., 2022) | energy combined with the norm of the feature residual outside the main principal subspace of training features | features + last layer |

Same protocol as the paper: corrected stage-matched split, six leave-one-class-out folds, threshold at the 95th percentile of validation scores, AUROC with session-bootstrap 95% CIs on the untouched test block, and false rejections split by fresh and spoiled known produce.

**Kaggle inputs:** `sentragrade-data` (with `spoilage_labels.csv`), the output of the leakage notebook (ResNet checkpoints), and the output of the ViT/DINOv2 notebook (heads). Turn on **GPU** and **Internet** (the transformer images are re-embedded once, ~5 min). The ResNet part needs no GPU.

In [ ]:
import os, glob, re, time, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
try:
    import timm
except ImportError:
    os.system("pip install -q timm"); import timm

# ---------------- config ----------------
SEED = 42
CLASSES = ["Banana", "Carrot", "Guava", "Indian_Gooseberry", "Mango", "Tomato"]
FOLDS = list(range(6))
PROTOCOLS = ["corrected"]          # only the corrected split is evaluated here
SPLIT_MODE, SPLIT_DIRECTION = "stage_matched", "forward"   # must match the leakage notebook
TRAIN_FRAC, BUFFER_DAYS, SESSION_GAP_MIN = 0.60, 1, 5
KNN_K = 50
REACT_PCT = 90
N_BOOT = 1000
TIMM_NAMES = {"vit_b16": "vit_base_patch16_224.orig_in21k", "dinov2_s14": "vit_small_patch14_dinov2.lvd142m"}
TIMM_KW = {"vit_b16": {}, "dinov2_s14": {"img_size": 224}}
QUICK = False

INPUT = "/kaggle/input"
def find_data_root():
    if os.environ.get("SENTRAGRADE_DATA_ROOT"):
        return Path(os.environ["SENTRAGRADE_DATA_ROOT"])
    hits = glob.glob(f"{INPUT}/**/manifest_full.csv", recursive=True)
    return Path(hits[0]).parent if hits else Path("~/sentragrade_data").expanduser()
DATA_ROOT = find_data_root()
def find_spoil_csv():
    if os.environ.get("SPOIL_CSV"):
        return Path(os.environ["SPOIL_CSV"])
    hits = glob.glob(f"{INPUT}/**/spoilage_labels.csv", recursive=True) + ["kaggle/spoilage_labels.csv"]
    return next((Path(h) for h in hits if os.path.exists(h)), None)
SPOIL_CSV = find_spoil_csv()
RESNET_DIR = os.environ.get("RESNET_CKPT_DIR", INPUT)
PROBE_DIR = os.environ.get("PROBE_DIR", INPUT)
RESNET_CKPTS = {int(re.search(r"fold(\d)", p).group(1)): p for p in glob.glob(f"{RESNET_DIR}/**/resnet50_corrected_fold*.pt", recursive=True)}
PROBES = {}
for p in glob.glob(f"{PROBE_DIR}/**/probe_*_corrected_fold*.pt", recursive=True):
    m = re.search(r"probe_(.+)_corrected_fold(\d)", os.path.basename(p))
    PROBES[(m.group(1), int(m.group(2)))] = p

OUT = Path("/kaggle/working/sentragrade_baselines") if Path("/kaggle/working").exists() else Path("outputs/sentragrade_baselines")
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if QUICK:
    FOLDS, N_BOOT = [0], 50
print("data:", DATA_ROOT, "| spoilage:", SPOIL_CSV, "| ResNet folds:", sorted(RESNET_CKPTS),
      "| probes:", sorted(PROBES), "| device:", DEVICE)
assert (DATA_ROOT / "manifest_full.csv").exists() and SPOIL_CSV is not None
assert RESNET_CKPTS or PROBES, "attach the leakage-notebook and/or the ViT/DINOv2-notebook outputs"

## 1. Splits (same code as the leakage notebook)

In [ ]:
man = pd.read_csv(DATA_ROOT / "manifest_full.csv", dtype={"date": str, "time": str})
man["path"] = [str(DATA_ROOT / "resized" / c / f) for c, f in zip(man.class_name, man.filename)]
missing = [p for p in man.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"
if SPOIL_CSV is not None:
    sp = pd.read_csv(SPOIL_CSV)
    man = man.drop(columns=[c for c in ["spoil_status"] if c in man.columns]).merge(sp, on=["class_name", "filename"], how="left")
    assert man.spoil_status.notna().all(), f"{man.spoil_status.isna().sum()} images have no spoilage label"
else:
    man["spoil_status"] = "unknown"

# capture sessions: a burst of ~17 photos; used as the resampling unit for the bootstrap
man["ts"] = pd.to_datetime(man.date + man.time.str.zfill(6), format="%Y%m%d%H%M%S")
man = man.sort_values(["class_name", "ts"]).reset_index(drop=True)
gap = man.groupby("class_name").ts.diff() > pd.Timedelta(minutes=SESSION_GAP_MIN)
man["session"] = man.class_name + "_" + gap.groupby(man.class_name).cumsum().astype(str)
key = dict(zip(man.class_name + "/" + man.filename, man.session))
spoil_key = dict(zip(man.class_name + "/" + man.filename, man.spoil_status))

def blocked_split(d):
    # whole capture days, in time order: train block, buffer day(s) dropped, then val and test blocks.
    # Small blocks (a stage with only 2-4 days) degrade gracefully: the buffer is skipped when it would
    # leave fewer than 2 days, and 2 days give train + test with no val.
    days = sorted(d.date.unique(), reverse=(SPLIT_DIRECTION == "reverse"))
    if len(days) == 1:
        return days, [], [], []
    if len(days) == 2:
        return days[:1], [], [], days[1:]
    n = d.groupby("date").size().reindex(days).values
    cum = np.cumsum(n) / n.sum()
    n_tr = int(np.clip((cum <= TRAIN_FRAC).sum(), 1, len(days) - 2))
    buf_n = BUFFER_DAYS if len(days) - n_tr - BUFFER_DAYS >= 2 else 0
    rest = days[n_tr + buf_n:]
    k = max(1, len(rest) // 2)
    return days[:n_tr], days[n_tr:n_tr + buf_n], rest[:k], rest[k:]

def split_class(d):
    if SPLIT_MODE == "time_blocked":
        return blocked_split(d)
    # stage-matched: every capture day is all fresh or all spoiled (checked), block each stage separately
    stage_of_day = d.groupby("date").spoil_status.agg(lambda x: x.mode().iloc[0])
    mixed = d.groupby("date").spoil_status.nunique()
    assert (mixed == 1).all(), f"{d.class_name.iloc[0]}: some days mix fresh and spoiled images"
    out = [[], [], [], []]
    for st in sorted(stage_of_day.unique()):
        part = blocked_split(d[d.date.isin(stage_of_day[stage_of_day == st].index)])
        for i in range(4):
            out[i] += list(part[i])
    return tuple(sorted(x) for x in out)

split_days, rows = {}, []
for c, d in man.groupby("class_name"):
    tr, buf, va, te = split_class(d)
    assert va and te, f"{c}: split left no val or no test days"
    split_days[c] = dict(train=tr, buffer=buf, val=va, test=te)
    fr = lambda days: f"{(d.date.isin(days) & (d.spoil_status == 'Not_spoiled')).sum()}/{(d.date.isin(days) & (d.spoil_status == 'Spoiled')).sum()}"
    rows.append({"class": c, "days": d.date.nunique(),
                 "train_days": ",".join(tr), "buffer": ",".join(buf),
                 "val_days": ",".join(va), "test_days": ",".join(te),
                 "train_n": d.date.isin(tr).sum(), "val_n": d.date.isin(va).sum(),
                 "test_n": d.date.isin(te).sum(), "dropped_n": d.date.isin(buf).sum(),
                 "train_fresh/spoiled": fr(tr), "val_fresh/spoiled": fr(va), "test_fresh/spoiled": fr(te)})
corrected_table = pd.DataFrame(rows)
display(corrected_table)
corrected_table.to_csv(OUT / "corrected_split_days.csv", index=False)

def tag(df):
    df = df.copy()
    df["path"] = [str(DATA_ROOT / "resized" / c / f) for c, f in zip(df.class_name, df.filename)]
    df["session"] = (df.class_name + "/" + df.filename).map(key)
    df["spoil_status"] = (df.class_name + "/" + df.filename).map(spoil_key)
    return df

def get_fold(protocol, k):
    held = CLASSES[k]
    known = [c for c in CLASSES if c != held]
    label_map = {c: i for i, c in enumerate(sorted(known))}
    ood = man[man.class_name == held]
    if protocol == "old":
        tr = tag(pd.read_csv(DATA_ROOT / "folds" / f"fold{k}_train.csv", dtype={"date": str}))
        va = tag(pd.read_csv(DATA_ROOT / "folds" / f"fold{k}_val.csv", dtype={"date": str}))
        te = va                    # the paper reports on val: reproduce that exactly
    else:
        kn = man[man.class_name != held]
        pick = lambda part: pd.concat([kn[(kn.class_name == c) & kn.date.isin(split_days[c][part])] for c in known])
        tr, va, te = pick("train"), pick("val"), pick("test")
        # checks: no capture day shared between blocks, buffer gap respected, test unseen
        for c in known:
            s = split_days[c]
            assert not (set(s["train"]) & set(s["val"])) and not (set(s["train"]) & set(s["test"])) and not (set(s["val"]) & set(s["test"]))
        assert set(tr.path).isdisjoint(te.path) and set(va.path).isdisjoint(te.path) and set(tr.path).isdisjoint(va.path)
    assert held not in set(tr.class_name) | set(va.class_name) | set(te.class_name)
    if QUICK:
        tr = tr.groupby("class_name").head(40); va = va.groupby("class_name").head(20)
        te = te.groupby("class_name").head(20) if protocol == "corrected" else va; ood = ood.head(60)
    return dict(train=tr, val=va, test=te, ood=ood, label_map=label_map, held=held)

summary = []
for p in PROTOCOLS:
    for k in FOLDS:
        f = get_fold(p, k)
        summary.append({"protocol": p, "fold": k, "held_out": f["held"], "train": len(f["train"]),
                        "val": len(f["val"]), "test": len(f["test"]) if p == "corrected" else "= val", "ood": len(f["ood"])})
display(pd.DataFrame(summary))

## 2. Re-embed the images with the frozen transformers (only if their heads are attached)

In [ ]:
class PathDS(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB"))

need = sorted(set().union(*[set(pd.concat([get_fold("corrected", k)[s] for s in ["train", "val", "test", "ood"]]).path) for k in FOLDS]))
EMB = {}
for name in sorted({b for b, _ in PROBES}):
    model = timm.create_model(TIMM_NAMES[name], pretrained=True, num_classes=0, **TIMM_KW[name]).to(DEVICE).eval()
    cfg = timm.data.resolve_data_config({}, model=model)
    tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                             transforms.Normalize(cfg["mean"], cfg["std"])])
    feats, t0 = [], time.time()
    with torch.no_grad():
        for x in DataLoader(PathDS(need, tf), batch_size=64, num_workers=NUM_WORKERS):
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                feats.append(model(x.to(DEVICE)).float().cpu())
    EMB[name] = (dict(zip(need, range(len(need)))), torch.cat(feats))
    print(f"{name}: {tuple(EMB[name][1].shape)} in {time.time()-t0:.0f}s")
    del model

## 3. Scores

In [ ]:
def vim_dim(d):
    return 1000 if d >= 2048 else (512 if d >= 768 else d // 2)   # principal-subspace sizes used by the ViM paper

def fit(Ftr, ytr, W, b):
    # everything estimated from training data only
    n = W.shape[0]
    mu = torch.stack([Ftr[ytr == c].mean(0) for c in range(n)])
    Xc = Ftr - mu[ytr]
    prec = torch.linalg.pinv(torch.cov(Xc.T.double())).float()              # shared covariance (Mahalanobis)
    clip = float(np.percentile(Ftr.numpy(), REACT_PCT))                    # ReAct threshold
    u = -(torch.linalg.pinv(W.double()) @ b.double()).float()              # ViM origin
    Z = (Ftr - u).double()
    eigval, eigvec = torch.linalg.eigh(Z.T @ Z / len(Z))
    NS = eigvec[:, torch.argsort(eigval, descending=True)][:, vim_dim(Ftr.shape[1]):].float()
    alpha = float((Ftr @ W.T + b).max(1).values.mean() / torch.linalg.norm((Ftr - u) @ NS, dim=1).mean())
    return dict(mu=mu, prec=prec, clip=clip, u=u, NS=NS, alpha=alpha, Trn=F.normalize(Ftr, dim=1), W=W, b=b)

def scores(P, Fx):
    W, b = P["W"], P["b"]
    L = Fx @ W.T + b
    lse = torch.logsumexp(L, 1)
    d = Fx[:, None, :] - P["mu"][None]
    maha = ((d @ P["prec"]) * d).sum(-1).min(1).values
    sims = F.normalize(Fx, dim=1) @ P["Trn"].T
    k = min(KNN_K, sims.shape[1])
    knn = -sims.topk(k, dim=1).values[:, -1]
    react = -torch.logsumexp(Fx.clamp(max=P["clip"]) @ W.T + b, 1)
    vim = P["alpha"] * torch.linalg.norm((Fx - P["u"]) @ P["NS"], dim=1) - lse
    out = {"MSP": -L.softmax(1).max(1).values, "Energy": -lse, "MaxLogit": -L.max(1).values,
           "Proto": torch.cdist(Fx, P["mu"]).min(1).values, "Mahalanobis": maha, "KNN": knn,
           "ReAct": react, "ViM": vim}
    return {m: v.numpy().astype(np.float64) for m, v in out.items()}

def fpr_at_95(neg, pos):
    return float((neg >= np.percentile(pos, 5)).mean())

def boot_auroc(neg, pos, ns, ps, rng):
    a_ = pd.Series(list(neg)).groupby(ns).apply(np.array).values
    b_ = pd.Series(list(pos)).groupby(ps).apply(np.array).values
    out = []
    for _ in range(N_BOOT):
        a = np.concatenate(a_[rng.integers(0, len(a_), len(a_))]); b = np.concatenate(b_[rng.integers(0, len(b_), len(b_))])
        out.append(roc_auc_score(np.r_[np.zeros(len(a)), np.ones(len(b))], np.r_[a, b]))
    return np.percentile(out, [2.5, 97.5])

## 4. Run every backbone and fold

In [ ]:
def backbone_data(name, k, f):
    lm = f["label_map"]; y = lambda df: torch.tensor(df.class_name.map(lm).values)
    if name == "resnet50":
        ck = torch.load(RESNET_CKPTS[k], map_location="cpu", weights_only=False)
        assert ck.get("split_mode") == SPLIT_MODE and ck["label_map"] == lm
        for s in ["train", "val", "test", "ood"]:
            assert len(ck[s]["features"]) == len(f[s]), f"fold {k} {s}: checkpoint does not match the rebuilt split"
        sd = ck["model_state_dict"]
        D = {s: ck[s]["features"].float() for s in ["train", "val", "test", "ood"]}
        return D, y(f["train"]), sd["fc.weight"].float(), sd["fc.bias"].float(), ck
    p = torch.load(PROBES[(name, k)], map_location="cpu", weights_only=False)
    assert p["label_map"] == lm
    idx, X = EMB[name]
    D = {s: X[[idx[q] for q in f[s].path]] for s in ["train", "val", "test", "ood"]}
    return D, y(f["train"]), p["head_state_dict"]["weight"].float(), p["head_state_dict"]["bias"].float(), None

rows = []
backbones = (["resnet50"] if RESNET_CKPTS else []) + sorted({b for b, _ in PROBES})
for name in backbones:
    for k in FOLDS:
        f = get_fold("corrected", k)
        D, ytr, W, b, ck = backbone_data(name, k, f)
        if ck is not None:   # sanity check: recomputed logits equal the stored ones
            assert torch.allclose(D["test"] @ W.T + b, ck["test"]["logits"].float(), atol=1e-2)
        P = fit(D["train"], ytr, W, b)
        S = {s: scores(P, D[s]) for s in ["val", "test", "ood"]}
        stage = f["test"].spoil_status.values; fresh, spoiled = stage == "Not_spoiled", stage == "Spoiled"
        rng = np.random.default_rng(SEED)
        for m in S["test"]:
            neg, pos, thr = S["test"][m], S["ood"][m], np.percentile(S["val"][m], 95)
            auc = lambda a: roc_auc_score(np.r_[np.zeros(len(a)), np.ones(len(pos))], np.r_[a, pos])
            lo, hi = boot_auroc(neg, pos, f["test"].session.values, f["ood"].session.values, rng)
            rows.append({"backbone": name, "fold": k, "held_out": f["held"], "method": m,
                         "AUROC": auc(neg), "AUROC_lo": lo, "AUROC_hi": hi, "FPR95": fpr_at_95(neg, pos),
                         "test_FRR_at_val_thr": float((neg > thr).mean()),
                         "test_FRR_fresh": float((neg[fresh] > thr).mean()), "test_FRR_spoiled": float((neg[spoiled] > thr).mean()),
                         "AUROC_vs_fresh_known": auc(neg[fresh]), "AUROC_vs_spoiled_known": auc(neg[spoiled])})
        r = pd.DataFrame(rows[-len(S["test"]):])
        print(f"{name:10s} fold {k} ({f['held']:17s}) " + "  ".join(f"{m} {a:.3f}" for m, a in zip(r.method, r.AUROC)))
res = pd.DataFrame(rows); res.to_csv(OUT / "baselines_per_fold.csv", index=False)

## 5. Summary

In [ ]:
ORDER = ["MSP", "Energy", "MaxLogit", "Proto", "Mahalanobis", "KNN", "ReAct", "ViM"]
summ = (res.groupby(["backbone", "method"])
          .agg(AUROC=("AUROC", "mean"), AUROC_sd=("AUROC", "std"), FPR95=("FPR95", "mean"),
               FRR_fresh=("test_FRR_fresh", "mean"), FRR_spoiled=("test_FRR_spoiled", "mean"),
               AUROC_vs_spoiled=("AUROC_vs_spoiled_known", "mean")).reset_index())
for c in ["FPR95", "FRR_fresh", "FRR_spoiled"]:
    summ[c] *= 100
summ["method"] = pd.Categorical(summ.method, ORDER, ordered=True)
summ = summ.sort_values(["backbone", "method"])
display(summ.round(4)); summ.to_csv(OUT / "baselines_summary.csv", index=False)
display(res.pivot_table(index=["backbone", "held_out"], columns="method", values="AUROC")[ORDER].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bbs = list(summ.backbone.unique()); x = np.arange(len(ORDER)); wb = 0.8 / len(bbs)
for ax, metric, lab in [(axes[0], "AUROC", "AUROC (higher is better)"), (axes[1], "FRR_spoiled", "% spoiled known produce rejected")]:
    for i, bb in enumerate(bbs):
        t = summ[summ.backbone == bb].set_index("method").reindex(ORDER)
        ax.bar(x + (i - (len(bbs) - 1) / 2) * wb, t[metric], wb, label=bb)
    ax.set_xticks(x); ax.set_xticklabels(ORDER, rotation=20); ax.set_title(lab); ax.legend()
axes[0].set_ylim(0.7, 1.0)
plt.tight_layout(); plt.savefig(OUT / "baselines.png", dpi=200); plt.show()
shutil.make_archive(str(OUT), "zip", OUT); print("Download:", str(OUT) + ".zip")